# Alpha Engine — 端到端训练流水线（API-first）

这个 notebook 现在只保留研究交互和少量训练代码；可复用逻辑迁移到：

- `ResearchSessionConfig`
- `prepare_training_frame`
- `run_10d_experiment`

流程：

```text
session_config.json + factor_selection.json
→ Qlib 加载 features/raw 10D returns
→ prepare_training_frame()
→ LightGBM 训练/预测
→ run_10d_experiment()
→ artifacts/evidence/notebook_10d_lab/<experiment>.json
```


In [ ]:
import sys, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init
from src.common.paths import ARTIFACTS_DIR
from src.research.notebook_lab_contracts import ResearchSessionConfig, CANONICAL_10D_RETURN_EXPR
from src.research.notebook_training_api import prepare_training_frame
from src.research.notebook_experiment_api import run_10d_experiment

cfg_path = ROOT / "data" / "session_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    MARKET, SYMBOLS, BENCHMARK = cfg["market"], cfg["symbols"], cfg["benchmark"]
    TRAIN_START, TRAIN_END = cfg["train_start"], cfg["train_end"]
    TEST_START, TEST_END = cfg["test_start"], cfg["test_end"]
else:
    MARKET = "us"
    SYMBOLS = ["AAPL", "NVDA", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "AVGO", "COST", "NFLX"]
    BENCHMARK = "QQQ"
    TRAIN_START, TRAIN_END = "2021-01-01", "2024-12-31"
    TEST_START, TEST_END = "2025-01-01", "2026-06-18"

config = ResearchSessionConfig(
    market=MARKET,
    symbols=SYMBOLS,
    benchmark=BENCHMARK,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_start=TEST_START,
    test_end=TEST_END,
    topk=15,
    label_type="raw_10d_return",
    experiment_id=f"{MARKET}_notebook_lgbm_10d",
    return_expression=CANONICAL_10D_RETURN_EXPR,
)

safe_qlib_init(build_qlib_init_cfg(None, market=MARKET))
from qlib.data import D
from qlib.contrib.data.loader import Alpha158DL

print(config)


In [ ]:
factor_selection_path = ROOT / config.factor_selection_path
selected_factors = []
if factor_selection_path.exists():
    factor_selection = json.loads(factor_selection_path.read_text(encoding="utf-8"))
    selected_factors = factor_selection.get("good_factors", [])
    print(f"Loaded factor_selection.json: {len(selected_factors)} good factors")
else:
    print("factor_selection.json not found; using Alpha158 subset + extras")

alpha = Alpha158DL.get_feature_config({
    "kbar": {},
    "price": {"windows": [0], "feature": ["OPEN", "HIGH", "LOW", "VWAP"]},
    "rolling": {},
})[0]
extras = [
    "$close/Ref($close,5)-1",
    "$close/Ref($close,10)-1",
    "$close/Ref($close,20)-1",
    "Std($close,10)",
    "$volume/Ref($volume,10)-1",
]
feature_exprs = list(alpha) + extras

X_all = D.features(SYMBOLS, feature_exprs, start_time=TRAIN_START, end_time=TEST_END)
if X_all.index.names == ["instrument", "datetime"]:
    X_all = X_all.swaplevel().sort_index()
X_all = X_all.fillna(0.0).replace([np.inf, -np.inf], 0.0)

safe_names = [
    str(expr).replace("$", "D").replace("/", "_d_").replace("(", "_").replace(")", "").replace(",", "_").replace(" ", "_")[:60]
    for expr in feature_exprs
]
X_all.columns = safe_names

if selected_factors:
    available = [name for name in selected_factors if name in X_all.columns]
    if available:
        X_all = X_all[available]
        print(f"Using selected factors available in feature matrix: {len(available)}")
    else:
        print("No selected factors matched loaded feature names; using full feature matrix")

raw = D.features(SYMBOLS, [config.return_expression], start_time=TRAIN_START, end_time=TEST_END)
raw_returns = raw.copy()
raw_returns.columns = ["return"]
if raw_returns.index.names == ["instrument", "datetime"]:
    raw_returns = raw_returns.swaplevel().sort_index()
raw_returns.attrs["provenance"] = "raw_forward_return"
raw_returns.attrs["horizon"] = 10
raw_returns.attrs["expression"] = config.return_expression

print(f"Features: {X_all.shape} | raw returns: {raw_returns.shape}")


In [ ]:
X, y = prepare_training_frame(X_all, raw_returns, label_type=config.label_type)

dates = X.index.get_level_values("datetime")
train_mask = (dates >= pd.Timestamp(TRAIN_START)) & (dates <= pd.Timestamp(TRAIN_END))
test_mask = (dates >= pd.Timestamp(TEST_START)) & (dates <= pd.Timestamp(TEST_END))

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_test = X.loc[test_mask]

PARAMS = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "seed": 42,
    "verbosity": -1,
}

print(f"Training samples={len(X_train):,}, test samples={len(X_test):,}, features={X_train.shape[1]}")
booster = lgb.train(PARAMS, lgb.Dataset(X_train, label=y_train), num_boost_round=500)
predictions = pd.DataFrame(booster.predict(X_test), index=X_test.index, columns=["score"])
predictions.attrs["provenance"] = "out_of_sample_model_prediction"

print(predictions.head())


In [ ]:
# Historical momentum baseline is a factor score, not a future return.
baseline = D.features(SYMBOLS, ["$close/Ref($close,10)-1"], start_time=TEST_START, end_time=TEST_END)
baseline.columns = ["score"]
if baseline.index.names == ["instrument", "datetime"]:
    baseline = baseline.swaplevel().sort_index()

test_returns = raw_returns.loc[
    (raw_returns.index.get_level_values("datetime") >= pd.Timestamp(TEST_START))
    & (raw_returns.index.get_level_values("datetime") <= pd.Timestamp(TEST_END))
].copy()
test_returns.attrs.update(raw_returns.attrs)

benchmark = None
try:
    bench_raw = D.features([BENCHMARK], [config.return_expression], start_time=TEST_START, end_time=TEST_END)
    if isinstance(bench_raw.index, pd.MultiIndex):
        benchmark = bench_raw.xs(BENCHMARK, level="instrument")
    else:
        benchmark = bench_raw
    benchmark.columns = ["benchmark"]
except Exception as exc:
    print(f"Benchmark unavailable: {exc}")

experiment = run_10d_experiment(
    config=config,
    candidates={
        "lgbm:regressor": predictions,
        "factor:historical_momentum_10d": baseline,
    },
    raw_returns=test_returns,
    benchmark_returns=benchmark,
    output_dir=ROOT / "artifacts" / "evidence" / "notebook_10d_lab",
)

summary = experiment["comparison_report"]["summary"]
print(json.dumps(summary, indent=2, default=str))


In [ ]:
artifact_dir = ARTIFACTS_DIR / "models" / config.experiment_id
artifact_dir.mkdir(parents=True, exist_ok=True)

model_path = artifact_dir / "model.pkl"
pred_path = artifact_dir / "predictions.parquet"

with model_path.open("wb") as handle:
    pickle.dump(booster, handle)
predictions.to_parquet(pred_path)

print(f"Saved model -> {model_path}")
print(f"Saved predictions -> {pred_path}")
print(f"Experiment report -> {experiment.get('artifact_path')}")
